# Examen B1 - RI
## Leandro Bravo
## Gr1CC

## Librerías:

In [32]:
import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter
import os
import matplotlib.pyplot as plt
import gensim.downloader as api
import sys
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from pathlib import Path
current_dir = Path.cwd()
examen_root = current_dir.parent
prepro_dir = examen_root / '05prepro'

if str(prepro_dir) not in sys.path:
    sys.path.insert(0, str(prepro_dir))
from prepro_func import ensure_nltk_resources, remove_special_characters, tokenize
ensure_nltk_resources()

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## Paso 1: Cargar Corpus

In [10]:
path = kagglehub.dataset_download("stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset")

print("Path to dataset files:", path)
print("Dataset files downloaded successfully")

Path to dataset files: C:\Users\leand\.cache\kagglehub\datasets\stefanoleone992\rotten-tomatoes-movies-and-critic-reviews-dataset\versions\1
Dataset files downloaded successfully


## Paso 2: Selección de campos textuales

### Cargar datasets

In [20]:
print("Cargando datasets...")
movies_path = f"{path}/rotten_tomatoes_movies.csv"
reviews_path = f"{path}/rotten_tomatoes_critic_reviews.csv"

movies_df = pd.read_csv(movies_path)
reviews_df = pd.read_csv(reviews_path)

# Unimos utilizando 'rotten_tomatoes_link' 
df_merged = pd.merge(reviews_df, movies_df[['rotten_tomatoes_link', 'movie_title', 'genres']], 
                     on='rotten_tomatoes_link', 
                     how='inner')

# Seleccionamos los campos necesarios
df = df_merged[['genres', 'movie_title','review_content','critic_name']].dropna(subset=['review_content']).reset_index(drop=True)
df.insert(0, 'Document ID', df.index.map(lambda i: f"Movie_{i}"))

corpus_df = df.copy()
corpus_df

Cargando datasets...


,Document ID,genres,movie_title,review_content,critic_name
0,Movie_0,"Action & Adventure, Comedy, Drama, Science Fic...",Percy Jackson & the Olympians: The Lightning T...,A fantasy adventure that fuses Greek mythology...,Andrew L. Urban
1,Movie_1,"Action & Adventure, Comedy, Drama, Science Fic...",Percy Jackson & the Olympians: The Lightning T...,"Uma Thurman as Medusa, the gorgon with a coiff...",Louise Keller
2,Movie_2,"Action & Adventure, Comedy, Drama, Science Fic...",Percy Jackson & the Olympians: The Lightning T...,With a top-notch cast and dazzling special eff...,NaN
3,Movie_3,"Action & Adventure, Comedy, Drama, Science Fic...",Percy Jackson & the Olympians: The Lightning T...,Whether audiences will get behind The Lightnin...,Ben McEachen
4,Movie_4,"Action & Adventure, Comedy, Drama, Science Fic...",Percy Jackson & the Olympians: The Lightning T...,What's really lacking in The Lightning Thief i...,Ethan Alter
...,...,...,...,...,...
1064104,Movie_1064104,"Classics, Drama",Zulu,A rousing reconstruction of the 1879 Battle of...,Joan Didion
1064105,Movie_1064105,"Action & Adventure, Art House & International,...",Zulu Dawn,"Seen today, it's not only a startling indictme...",Ken Hanke
1064106,Movie_1064106,"Action & Adventure, Art House & International,...",Zulu Dawn,A rousing visual spectacle that's a prequel of...,Dennis Schwartz
1064107,Movie_1064107,"Action & Adventure, Art House & International,...",Zulu Dawn,"A simple two-act story: Prelude to war, and th...",Christopher Lloyd


## Paso 3: Preprocesamiento

In [27]:
# Minúsculas
corpus_df['cleaned_review'] = corpus_df['review_content'].apply(lambda x: x.lower())

In [28]:
# Eliminamos caracteres especiales
corpus_df['cleaned_review'] = corpus_df['review_content'].apply(lambda x: remove_special_characters(x))

## Paso 4: Embeddings

### Cargar modelo

In [34]:
model = SentenceTransformer('all-MiniLM-L6-v2')

C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\leand\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading w

In [43]:
# Toma una muestra aleatoria de 10000 filas
subset_df = corpus_df.sample(n=10000, random_state=42).reset_index(drop=True)
subset_reviews = subset_df['cleaned_review'].tolist()
# Genera los embeddings para la muestra
embeddings = model.encode(subset_reviews, show_progress_bar=True)

Batches: 100%|██████████| 313/313 [00:23<00:00, 13.31it/s]


## Paso 5: Procesamiento de consultas

In [59]:
def recuperacion(query, embeddings, corpus_df, top_k=10):
    queries_lower = [query.lower() for query in query]
    query = remove_special_characters(queries_lower)
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings).flatten()
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    df_resultados = corpus_df.iloc[top_indices].copy()
    
    ranking_df = pd.DataFrame({
        'Ranking': range(1, len(top_indices) + 1),
        'ID documento': df_resultados['id'] if 'id' in corpus_df.columns else df_resultados.index,
        'Título película': df_resultados['movie_title'],
        'Fragmento de texto': df_resultados['cleaned_review'],
        'Similitud': similarities[top_indices]
    })
    
    return ranking_df.reset_index(drop=True)

## Paso 6: Benchmark de consultas

In [60]:
queries = [ 
    "science fiction movie with advanced technology",
    "romantic story with emotional relationships",
    "action movie with intense fight scenes",
    "horror film that creates fear and suspense",
    "visually impressive movie with weak storyline",
    "emotionally moving performance by the lead actor",
    "predictable plot but entertaining experience",
    "movie praised by critics but unpopular with audiences"
]

resumen_st = []

for query in queries:
    df_resultados = recuperacion(queries, embeddings, corpus_df, top_k=10)
    resumen_st.append({
        'query': query,
        'resultados': df_resultados
    })
    
    print(f"\n Query: '{query}'")

    for _, fila in df_resultados.iterrows():
        ranking = fila['Ranking']
        score = fila['Similitud']
        titulo = fila['Título película']
        texto = fila['Fragmento de texto']
        
        print(f"[{ranking}] Película: {titulo} (Score: {score:.4f})")
        print(f"    Texto: {texto[:200]}...\n")

TypeError: expected string or bytes-like object, got 'list'